In [10]:
!uv add stopwordsiso

Resolved 285 packages in 3.83s
Prepared 1 package in 54ms
Installed 1 package in 23ms
 + stopwordsiso==0.6.1


In [1]:
import pandas as pd

In [2]:
df_normar = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\normal_비율맞춘_학습데이터셋.csv')
df_phishing = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_total.csv')

In [3]:
df_normar.head()

,file_name,category,subcategory,speaker,text
0,Empathy_기쁨_부모자녀_조손_111.json,기쁨,부모자녀/조손,0,"엄마, 저 오늘 정말 신기한 일 있었어요! 제가 오늘 집에 오는 길에 누구를 만났는..."
1,Empathy_기쁨_부모자녀_조손_111.json,기쁨,부모자녀/조손,1,누구를 만났길래 그렇게 신이 났니? 엄마도 아는 사람일까? 궁금하구나.
2,Empathy_기쁨_부모자녀_조손_111.json,기쁨,부모자녀/조손,0,민지라고 기억하세요? 저랑 친했던 친구였어요.
3,Empathy_기쁨_부모자녀_조손_111.json,기쁨,부모자녀/조손,1,"어머, 초등학교 3학년 때 같은 반했던 친구 아니니? 우리 집에도 놀러 왔던 걸로 ..."
4,Empathy_기쁨_부모자녀_조손_111.json,기쁨,부모자녀/조손,0,"네, 맞아요! 민지도 엄마를 기억하더라고요. 잘 지내시냐고 안부 물었어요."


In [4]:
df_phishing.head()

,file_name,phishing_type,speaker,text
0,phishing_000,가족지인사칭형,0,"여보세요, OOO야. 나 아빠야."
1,phishing_000,가족지인사칭형,1,아빠? 목소리가 좀 이상한데요.
2,phishing_000,가족지인사칭형,0,아빠 핸드폰이 고장 나서 친구 폰으로 걸었어. 급하게 돈이 필요해서.
3,phishing_000,가족지인사칭형,1,무슨 일인데 그렇게 급해요?
4,phishing_000,가족지인사칭형,0,교통사고가 나서 병원비가 필요해. 믿고 바로 보내줄 수 있지?


In [5]:
df_phishing = df_phishing.rename(columns={'phishing_type': 'category'})

In [6]:
df_phishing['data_type'] = '피싱'
df_normar['data_type'] = '일반'

In [7]:
df_normer = df_normar.drop(columns=['subcategory'])

In [8]:
df_normer.head()

,file_name,category,speaker,text,data_type
0,Empathy_기쁨_부모자녀_조손_111.json,기쁨,0,"엄마, 저 오늘 정말 신기한 일 있었어요! 제가 오늘 집에 오는 길에 누구를 만났는...",일반
1,Empathy_기쁨_부모자녀_조손_111.json,기쁨,1,누구를 만났길래 그렇게 신이 났니? 엄마도 아는 사람일까? 궁금하구나.,일반
2,Empathy_기쁨_부모자녀_조손_111.json,기쁨,0,민지라고 기억하세요? 저랑 친했던 친구였어요.,일반
3,Empathy_기쁨_부모자녀_조손_111.json,기쁨,1,"어머, 초등학교 3학년 때 같은 반했던 친구 아니니? 우리 집에도 놀러 왔던 걸로 ...",일반
4,Empathy_기쁨_부모자녀_조손_111.json,기쁨,0,"네, 맞아요! 민지도 엄마를 기억하더라고요. 잘 지내시냐고 안부 물었어요.",일반


In [9]:
df_phishing.head()

,file_name,category,speaker,text,data_type
0,phishing_000,가족지인사칭형,0,"여보세요, OOO야. 나 아빠야.",피싱
1,phishing_000,가족지인사칭형,1,아빠? 목소리가 좀 이상한데요.,피싱
2,phishing_000,가족지인사칭형,0,아빠 핸드폰이 고장 나서 친구 폰으로 걸었어. 급하게 돈이 필요해서.,피싱
3,phishing_000,가족지인사칭형,1,무슨 일인데 그렇게 급해요?,피싱
4,phishing_000,가족지인사칭형,0,교통사고가 나서 병원비가 필요해. 믿고 바로 보내줄 수 있지?,피싱


In [ ]:
import os
import time
import logging
from dataclasses import dataclass
from typing import List

import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# tqdm를 pandas에 등록하여 progress_apply 사용
tqdm.pandas(desc="문서 토큰화 진행")

# 0. 설정 관리
@dataclass
class Config:
    INPUT_DF_NORMAL: str = "df_normer"       # DataFrame 객체명 또는 파일 경로
    INPUT_DF_PHISHING: str = "df_phishing"   # DataFrame 객체명 또는 파일 경로
    MERGED_PATH: str = "merged_data.csv"
    GROUPED_PATH: str = "grouped_cat_speaker.csv"
    RESULT_PATH: str = "tfidf_cat_speaker_final.csv"
    TOKENIZED_PATH: str = "tokenized_texts.csv"     # 형태소 분석 결과 저장 경로 추가
    LOG_PATH: str = "tfidf_process.log"

cfg = Config()

# 1. 로깅 설정
logging.basicConfig(
    filename=cfg.LOG_PATH,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("[%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)

def print_progress(step: str):
    logging.info(step)

def save_progress(df: pd.DataFrame, filename: str):
    try:
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        logging.info(f"저장됨: {filename}")
    except Exception as e:
        logging.error(f"파일 저장 실패 ({filename}): {e}")
        raise

def validate_dataframe(df: pd.DataFrame, required_cols: List[str], name: str):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msg = f"{name}에 필수 컬럼 누락: {missing}"
        logging.error(msg)
        raise ValueError(msg)
    if df.empty:
        msg = f"{name}가 비어 있습니다."
        logging.error(msg)
        raise ValueError(msg)
    logging.info(f"{name} 검증 완료 (행: {len(df)}, 열: {list(df.columns)})")

# 2. 데이터 준비
try:
    validate_dataframe(df_normer, ["category", "speaker", "text"], "일반 대화 데이터(df_normer)")
    validate_dataframe(df_phishing, ["category", "speaker", "text"], "피싱 데이터(df_phishing)")

    if "phishing_type" in df_phishing.columns:
        df_phishing = df_phishing.rename(columns={"phishing_type": "category"})
    print_progress("피싱 데이터 category 컬럼명 통일 완료")

    start = time.time()
    df_all = pd.concat([df_normer, df_phishing], ignore_index=True)
    print_progress("일반/피싱 데이터프레임 합치기 완료")
    save_progress(df_all, cfg.MERGED_PATH)
    logging.info(f"데이터 병합 소요: {time.time() - start:.2f}s")
except Exception as e:
    logging.critical(f"데이터 준비 단계 실패: {e}")
    raise

# 3. 형태소 분석기 및 토크나이저 정의
try:
    kiwi = Kiwi()
    def kiwi_tokenizer_with_lemma(text: str) -> List[str]:
        tokens = kiwi.tokenize(text)
        keywords = []
        for token in tokens:
            if token.tag in ["NNG", "NNP"]:
                keywords.append(token.form)
            elif token.tag in ["VV", "VA"]:
                keywords.append(token.lemma)
        return keywords
    print_progress("Kiwi 토크나이저 함수 정의 완료")
except Exception as e:
    logging.critical(f"토크나이저 정의 실패: {e}")
    raise

# 4. 카테고리+화자별 문서 단위 생성 (category 유지)
try:
    start = time.time()
    df_all["cat_speaker"] = df_all["category"].astype(str) + "_" + df_all["speaker"].astype(str)
    grouped = (
        df_all
        .groupby(["cat_speaker", "category"])["text"]
        .apply(lambda segs: " ".join(segs))
        .reset_index()
    )
    print_progress("카테고리+화자별 그룹화 및 문서 생성 완료")
    save_progress(grouped, cfg.GROUPED_PATH)
    logging.info(f"그룹화 소요: {time.time() - start:.2f}s")
except Exception as e:
    logging.critical(f"그룹화 단계 실패: {e}")
    raise

# 5. 한국어 불용어 리스트 로드
import stopwordsiso
korean_stopwords = stopwordsiso.stopwords("ko")

# 6. 형태소 분석(토큰화) 결과 별도 저장
try:
    print_progress("형태소 분석(토큰화) 결과 저장 시작")
    grouped["tokenized_text"] = grouped["text"].progress_apply(
        lambda txt: " ".join([t for t in kiwi_tokenizer_with_lemma(txt) if t not in korean_stopwords])
    )
    save_progress(grouped[["cat_speaker", "category", "tokenized_text"]], cfg.TOKENIZED_PATH)
    print_progress(f"형태소 분석(토큰화) 결과 별도 저장 완료: {cfg.TOKENIZED_PATH}")
except Exception as e:
    logging.critical(f"형태소 분석(토큰화) 결과 저장 실패: {e}")
    raise

# # 7. TF-IDF 벡터라이저 적용 (불용어 처리 및 진행 로그 추가)
# try:
#     # korean_stopwords는 set 타입이므로 list로 변환하거나
#     # 이미 grouped["tokenized_text"] 단계에서 불용어를 제거했다면 None으로 설정하세요.
#     vectorizer = TfidfVectorizer(
#         tokenizer=kiwi_tokenizer_with_lemma,
#         stop_words=list(korean_stopwords)  # ← set → list 변환
#         # stop_words=None  # ← 이미 tokenized_text에서 불용어를 제거했다면 이 옵션 사용
#     )
#     print_progress("TF-IDF 벡터라이저(불용어 처리 포함) 초기화 완료")

#     logging.info("TF-IDF 계산 시작")
#     start_tfidf = time.time()
#     tfidf_matrix = vectorizer.fit_transform(grouped["tokenized_text"])
#     logging.info(f"TF-IDF 계산 완료 (소요: {time.time() - start_tfidf:.2f}s)")
# except Exception as e:
#     logging.critical(f"TF-IDF 계산 실패: {e}")
#     raise

# # 8. 결과 변환 및 저장
# try:
#     feature_names = vectorizer.get_feature_names_out()
#     df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
#     print_progress("TF-IDF 결과 DataFrame 변환 완료")

#     df_result = pd.concat(
#         [grouped.reset_index(drop=True), df_tfidf.reset_index(drop=True)],
#         axis=1
#     )
#     print_progress("카테고리+화자별 문서와 TF-IDF 결과 합치기 완료")
#     save_progress(df_result, cfg.RESULT_PATH)
# except Exception as e:
#     logging.critical(f"결과 변환/저장 단계 실패: {e}")
#     raise


c:\Users\user\Desktop\woogawooga\woogawooga_project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 일반 대화 데이터(df_normer) 검증 완료 (행: 194240, 열: ['file_name', 'category', 'speaker', 'text', 'data_type'])
[INFO] 피싱 데이터(df_phishing) 검증 완료 (행: 44519, 열: ['file_name', 'category', 'speaker', 'text', 'data_type'])
[INFO] 피싱 데이터 category 컬럼명 통일 완료
[INFO] 일반/피싱 데이터프레임 합치기 완료
[INFO] 저장됨: merged_data.csv
[INFO] 데이터 병합 소요: 0.69s
[INFO] Kiwi 토크나이저 함수 정의 완료
[INFO] 카테고리+화자별 그룹화 및 문서 생성 완료
[INFO] 저장됨: grouped_cat_speaker.csv
[INFO] 그룹화 소요: 0.43s
c:\Users\user\Desktop\woogawooga\woogawooga_project\.venv\lib\site-packages\stopwordsiso\_core.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025

InvalidParameterError: The 'stop_words' parameter of TfidfVectorizer must be a str among {'english'}, an instance of 'list' or None. Got {'참', '왜', '할망정', '훨씬', '하', '（', '만일', '삼', '거바', '뿐이다', '과', '아이야', '구체적으로', '바로', '이렇구나', '따라', '보는데서', '오직', '것과 같이', '동시에', '조금', '논하지 않다', '아이', '아니라면', '더불어', '어찌됏든', '대해 말하자면', '기타', '、', '하지 않도록', '흥', '아이쿠', '-', '갖고말하자면', '비추어 보아', '어찌됏어', '전후', '만이 아니다', '으로써', '일것이다', '바꾸어말하면', '로써', '에 가서', '2', '대하여', '시작하여', '(', '여덟', '만약', '옆사람', '휴', '～', '：', '￥', '어떠한', '않기 위하여', '|', '약간', '예하면', '거의', '각각', '관한', '그런데', '륙', '의해', '자', '저희', '하물며', '＞', '·', '하기 위하여', '정도에 이르다', '어디', '아이구', '그렇지', '봐라', '및', '제외하고', '잇따라', '형식으로 쓰여', '이상', '관계없이', '그치지 않다', '오히려', '이렇게되면', '하여금', '팔', '참나', '하는것도', '쾅쾅', '얼마', '두번째로', '꽈당', '그', '알 수 있다', '하기만 하면', '당장', '딩동', '바꾸어서 말하면', '와르르', '예를 들자면', '앗', '이천칠', '하느니', '어느때', '이렇게 많은 것', '조차', '때문에', '어느쪽', '만 못하다', '매번', '다음', '예', '향해서', '뚝뚝', '콸콸', '어째서', '8', '된바에야', '그렇지만', '결국', '고로', '좀', '《', '하하',

In [6]:
import pandas as pd
import logging
import time

# 간단한 print_progress 함수 정의 (원래는 로깅 함수였던 듯)
def print_progress(msg):
    print(msg)

# 로깅 설정 (간단히 콘솔 출력용)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

tokenized_path = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\tokenized_texts.csv"

try:
    grouped = pd.read_csv(tokenized_path, encoding="utf-8-sig")
    print_progress(f"토큰화 데이터 로드 완료: {tokenized_path}")
except Exception as e:
    logging.critical(f"토큰화 데이터 로드 실패: {e}")
    raise

from sklearn.feature_extraction.text import TfidfVectorizer

try:
    vectorizer = TfidfVectorizer(
        tokenizer=None,
        preprocessor=None,
        token_pattern=r"(?u)\b\w+\b",
        stop_words=None
    )
    print_progress("TF-IDF 벡터라이저 초기화 완료")

    logging.info("TF-IDF 계산 시작")
    start_tfidf = time.time()
    tfidf_matrix = vectorizer.fit_transform(grouped["tokenized_text"])
    logging.info(f"TF-IDF 계산 완료 (소요: {time.time() - start_tfidf:.2f}s)")
except Exception as e:
    logging.critical(f"TF-IDF 계산 실패: {e}")
    raise

try:
    feature_names = vectorizer.get_feature_names_out()
    df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
    print_progress("TF-IDF 결과 DataFrame 변환 완료")

    df_result = pd.concat(
        [grouped.reset_index(drop=True), df_tfidf.reset_index(drop=True)],
        axis=1
    )
    print_progress("카테고리+화자별 문서와 TF-IDF 결과 합치기 완료")

    # save_progress 함수가 없다면 간단히 csv 저장 코드로 대체
    result_path = "tfidf_cat_speaker_final.csv"
    df_result.to_csv(result_path, index=False, encoding="utf-8-sig")
    print_progress(f"결과 저장 완료: {result_path}")

except Exception as e:
    logging.critical(f"결과 변환/저장 단계 실패: {e}")
    raise


토큰화 데이터 로드 완료: C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\tokenized_texts.csv


2025-07-15 16:48:18 [INFO] TF-IDF 계산 시작


TF-IDF 벡터라이저 초기화 완료


2025-07-15 16:48:18 [INFO] TF-IDF 계산 완료 (소요: 0.55s)


TF-IDF 결과 DataFrame 변환 완료
카테고리+화자별 문서와 TF-IDF 결과 합치기 완료
결과 저장 완료: tfidf_cat_speaker_final.csv


In [ ]:

# 7. 머신러닝 학습용 데이터 준비
# '일반'과 '피싱'만 필터링
df_result = df_result[df_result["category"].isin(["일반", "피싱"])].copy()
df_result["category_label"] = df_result["category"].map({"일반": 0, "피싱": 1})

# TF-IDF 벡터와 라벨 추출 (인덱스 맞춤)
X = df_tfidf.loc[df_result.index].values
y = df_result["category_label"].values

# 8. 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 9. 분류 모델 학습
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 10. 예측 및 평가
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["일반", "피싱"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

# 11. 새로운 문장 예측 함수
def predict_sentence(text: str):
    tokens = kiwi_tokenizer_with_lemma(text)
    vec = vectorizer.transform([" ".join(tokens)]).toarray()
    proba = model.predict_proba(vec)[0, 1]
    if proba < 0.3:
        return "일반대화", proba
    elif proba < 0.7:
        return "보류 (2차 판정 필요)", proba
    else:
        return "보이스피싱", proba

# 사용 예시
sample = "고객님, 계좌 확인이 필요하니 로그인 해주세요."
label, score = predict_sentence(sample)
print(f"문장: {sample}\n예측: {label} (확률: {score:.2f})")

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [15]:
# 카테고리 필터 후 인덱스 재정렬
df_result = df_result[df_result["category"].isin(["일반", "피싱"])].reset_index(drop=True)
df_result["category_label"] = df_result["category"].map({"일반": 0, "피싱": 1})

In [16]:
df_tfidf = df_tfidf.reset_index(drop=True)

In [19]:
df_result = df_result[df_result["category"].isin(["일반", "피싱"])].reset_index(drop=True)
print("필터 후 행 개수:", len(df_result))

필터 후 행 개수: 0


In [22]:
grouped["category"] = grouped["category"].str.strip()

In [24]:
df_result = df_result[df_result["category"].isin(["일반", "피싱"])].reset_index(drop=True)
print("필터 후 행 개수:", len(df_result))
print("남은 카테고리 값:", df_result["category"].unique())

필터 후 행 개수: 0
남은 카테고리 값: []


In [74]:
df_tfidf.head()

,(2,11번가,19금,가게,가격,가격대,가결,가계,가계부,가고파,...,힐스테이트,힘,힘내다,힘닿다,힘들다,힘들어지다,힘쓰다,힘차다,힙합,힜
0,0.0,0.0,0.0,0.0,0.000857,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.001399,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.004524,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000863,0.0,0.0,0.004431,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.002919,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [ ]:
# cat_speaker, text 등 메타데이터 컬럼을 제외한 TF-IDF 값만 추출
# 만약 df_tfidf에 메타데이터 컬럼이 없고, 모두 토큰 컬럼이라면 바로 sum 사용
tfidf_sum = df_tfidf.sum(axis=1)
df_sorted = df_tfidf.copy()
df_sorted['tfidf_sum'] = tfidf_sum

# tfidf_sum 기준 내림차순 정렬
df_sorted = df_sorted.sort_values(by='tfidf_sum', ascending=False)

# 상위 5개 결과 확인
print(df_sorted.head())


In [2]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 환경
plt.rcParams['axes.unicode_minus'] = False

top_n = 60

# 1. 일반 대화만 필터링
general_mask = df_result['binary_category'] == '일반'
general_df_tfidf = df_tfidf.loc[general_mask]

# 2. 각 단어(피처)별 최대 TF-IDF 값 계산
max_tfidf_general = general_df_tfidf.max(axis=0).sort_values(ascending=False)[:top_n]

# 3. 시각화
plt.figure(figsize=(12, 6))
max_tfidf_general.plot(kind='bar', color='mediumseagreen')
plt.title('일반 대화에서 TF-IDF 값이 높은 상위 60개 단어')
plt.xlabel('단어')
plt.ylabel('최대 TF-IDF 값')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


NameError: name 'df_result' is not defined

In [3]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 사용 시
plt.rcParams['axes.unicode_minus'] = False

top_n = 60  # 추출할 단어 개수

# 1. 피싱 데이터만 필터링하여 TF-IDF 행렬 준비
phishing_mask = df_result['binary_category'] == '피싱'
phishing_df_tfidf = df_tfidf.loc[phishing_mask]

# 2. 각 단어(피처)별 최대 TF-IDF 값 계산
max_tfidf_phishing = phishing_df_tfidf.max(axis=0).sort_values(ascending=False)[:top_n]

# 3. 시각화
plt.figure(figsize=(12, 6))
max_tfidf_phishing.plot(kind='bar', color='skyblue')
plt.title('피싱 대화에서 TF-IDF 값이 높은 상위 60개 단어')
plt.xlabel('단어')
plt.ylabel('최대 TF-IDF 값')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

NameError: name 'df_result' is not defined

In [26]:
df_result = pd.read_csv(r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\tfidf_cat_speaker_final.csv")

In [28]:
df_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Columns: 18474 entries, cat_speaker to 힜
dtypes: float64(18470), object(4)
memory usage: 6.8+ MB
